# 07 — Progressives Fine-Tuning mit Kreuzvalidierung

Wiederholung von Notebook 06 mit fuenffacher, subjektunabhaengiger
Kreuzvalidierung.

**Warum diese Wiederholung noetig ist**

Notebook 06 verwendete eine einzelne Aufteilung. Deren Testsatz
enthielt lediglich 36 Sturzfenster. Die dort berichteten Werte
lauten:

| Strategie | TP | Se | 95 %-KI |
|---|---|---|---|
| A Zero-Shot | 7 | 0,194 | [0,098; 0,350] |
| B Direkt | 10 | 0,278 | [0,158; 0,440] |
| C2 Progressiv | 12 | 0,333 | [0,202; 0,497] |

Der Unterschied zwischen B und C2 betraegt **zwei richtig erkannte
Fenster**. Die Konfidenzintervalle ueberlappen vollstaendig; die
Aussage ist so nicht belastbar.

Hier wird daher ueber alle Probanden kreuzvalidiert. Zusaetzlich
werden beide Strategien **gepaart** verglichen: Da sie auf
denselben Falten bewertet werden, heben sich Schwankungen zwischen
den Falten heraus.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

!wget -q https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py -O /content/fall_lib.py
sys.path.insert(0, "/content")
import fall_lib as fl

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"

def hole(name, ziel):
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        vorhanden = "\n    ".join(p.name for p in sorted(Path(BASE).iterdir()))
        raise FileNotFoundError(
            f"'{name}' nicht gefunden unter {BASE}\n"
            f"  Vorhandene Ordner:\n    {vorhanden}")
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel

from tensorflow import keras
if not Path(MODEL).exists():
    raise FileNotFoundError(f"{MODEL} fehlt - bitte 01 ausfuehren.")
basis = keras.models.load_model(MODEL, compile=False)
basis.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
print(f"Basismodell: {basis.count_params():,} Parameter")

Mounted at /content/drive
Basismodell: 61,601 Parameter


In [2]:
N_FOLDS = 5
ZIEL_POSITION = "ANKLE"

run = fl.new_run("07_progressiv_cv")

kf = fl.load_kfall(hole("KFALL/sensor_data_new", "/content/KFall"),
                   label_dir=hole("KFALL/label_data_new",
                                  "/content/KFall_labels"))
uma = fl.load_umafall(
    hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
         "/content/UMAFall"), position=ZIEL_POSITION)

X_kf, y_kf, _, _ = fl.build_windows(kf, mode="standard")
X_kf = fl.znorm(X_kf)

X, y, g, _ = fl.build_windows(uma, mode="standard")
X = fl.znorm(X)
print(f"\nZiel: {len(y)} Fenster, {int(y.sum())} Stuerze, "
      f"{len(np.unique(g))} Probanden")

[Lauf] /content/drive/MyDrive/ML_Fall_Detection/results/07_progressiv_cv_2026-08-26_1218
[ok] KFALL/sensor_data_new kopiert in 166s
[ok] KFALL/label_data_new kopiert in 11s
[KFall] 5075 Trials (2346 Sturz, 2729 ADL), 32 Probanden
[KFall] Sturz-Tasks T20-T34 | 2346 Aufprallzeitpunkte aus Labeldateien
[ok] UMAFall Fall Detection/UMAFall_Dataset_corrected kopiert in 34s
[UMAFall ANKLE] 643 Trials (157 Sturz, 486 ADL), 19 Probanden

Ziel: 6486 Fenster, 157 Stuerze, 19 Probanden


## KFall-Zwischenstufe

Die KFall-Anpassung haengt nicht von der jeweiligen Falte ab, da die
KFall-Probanden ohnehin von den UMAFall-Probanden verschieden sind.
Sie wird daher einmal berechnet und in allen Falten wiederverwendet.

In [3]:
print("Fine-Tuning auf KFall (einmalig) ...")
t = time.time()
m_kfall = fl.finetune(basis, X_kf, y_kf)
print(f"  fertig in {time.time()-t:.0f}s")

Fine-Tuning auf KFall (einmalig) ...
  fertig in 60s


## Kreuzvalidierung

In [4]:
splits = fl.group_kfold_splits(g, y, n_splits=N_FOLDS)

falten = {"A_zero_shot": [], "B_direkt": [],
          "C1_nach_kfall": [], "C2_progressiv": []}

for i, (tr, te) in enumerate(splits, 1):
    n_f = int(y[te].sum())
    print(f"\nFalte {i}/{N_FOLDS}: Test {len(te)} Fenster, "
          f"{n_f} Stuerze, {len(np.unique(g[te]))} Probanden")
    if n_f == 0:
        print("  keine Stuerze im Testsatz - Falte uebersprungen")
        continue

    falten["A_zero_shot"].append(
        fl.evaluate(basis, X[te], y[te], "  A Zero-Shot"))
    falten["C1_nach_kfall"].append(
        fl.evaluate(m_kfall, X[te], y[te], "  C1 nach KFall"))

    m_direkt = fl.finetune(basis, X[tr], y[tr])
    falten["B_direkt"].append(
        fl.evaluate(m_direkt, X[te], y[te], "  B Direkt"))

    m_prog = fl.finetune(m_kfall, X[tr], y[tr])
    falten["C2_progressiv"].append(
        fl.evaluate(m_prog, X[te], y[te], "  C2 Progressiv"))


Falte 1/5: Test 1400 Fenster, 40 Stuerze, 4 Probanden
    A Zero-Shot               Se=0.325  Sp=0.976  Acc=0.957  FP=   33  FN=  27
    C1 nach KFall             Se=0.175  Sp=0.990  Acc=0.967  FP=   13  FN=  33


    B Direkt                  Se=0.425  Sp=0.968  Acc=0.952  FP=   44  FN=  23


    C2 Progressiv             Se=0.500  Sp=0.968  Acc=0.955  FP=   43  FN=  20

Falte 2/5: Test 1102 Fenster, 29 Stuerze, 4 Probanden
    A Zero-Shot               Se=0.276  Sp=0.961  Acc=0.943  FP=   42  FN=  21
    C1 nach KFall             Se=0.276  Sp=0.982  Acc=0.964  FP=   19  FN=  21
    B Direkt                  Se=0.379  Sp=0.956  Acc=0.941  FP=   47  FN=  18
    C2 Progressiv             Se=0.448  Sp=0.950  Acc=0.936  FP=   54  FN=  16

Falte 3/5: Test 2012 Fenster, 55 Stuerze, 4 Probanden
    A Zero-Shot               Se=0.145  Sp=0.983  Acc=0.960  FP=   33  FN=  47
    C1 nach KFall             Se=0.127  Sp=0.988  Acc=0.964  FP=   24  FN=  48
    B Direkt                  Se=0.455  Sp=0.974  Acc=0.960  FP=   51  FN=  30
    C2 Progressiv             Se=0.600  Sp=0.976  Acc=0.966  FP=   47  FN=  22

Falte 4/5: Test 1395 Fenster, 27 Stuerze, 4 Probanden
    A Zero-Shot               Se=0.333  Sp=0.971  Acc=0.958  FP=   40  FN=  18
    C1 nach KFall             Se=0.148  Sp=0.

## Zusammengefasstes Ergebnis

Aggregiert wird ueber die aufsummierten Konfusionsmatrizen, nicht
ueber die Mittelwerte der Falten -- sonst wuerden unterschiedlich
grosse Falten gleich stark gewichtet.

In [5]:
gepoolt = {k: fl.pool_folds(v) for k, v in falten.items() if v}

print(f"Ueber {N_FOLDS} Falten zusammengefasst:\n")
namen = {"A_zero_shot": "A  Zero-Shot",
         "B_direkt": "B  Direkt",
         "C1_nach_kfall": "C1 nach KFall",
         "C2_progressiv": "C2 Progressiv"}
for k, n in namen.items():
    if k in gepoolt:
        fl.print_pooled(n, gepoolt[k])

print(f"\n{'Strategie':<16} {'Se je Falte'}")
print("-"*52)
for k, n in namen.items():
    if k in gepoolt:
        print(f"{n:<16} {gepoolt[k]['Se_per_fold']}")

Ueber 5 Falten zusammengefasst:

  A  Zero-Shot           Se=0.255 [0.193, 0.328]  Sp=0.974  (TP=40, FN=117, FP=165)
  B  Direkt              Se=0.427 [0.352, 0.505]  Sp=0.967  (TP=67, FN=90, FP=211)
  C1 nach KFall          Se=0.172 [0.121, 0.239]  Sp=0.987  (TP=27, FN=130, FP=82)
  C2 Progressiv          Se=0.522 [0.445, 0.599]  Sp=0.965  (TP=82, FN=75, FP=220)

Strategie        Se je Falte
----------------------------------------------------
A  Zero-Shot     [np.float64(0.325), np.float64(0.2759), np.float64(0.1455), np.float64(0.3333), np.float64(0.3333)]
B  Direkt        [np.float64(0.425), np.float64(0.3793), np.float64(0.4545), np.float64(0.4444), np.float64(0.3333)]
C1 nach KFall    [np.float64(0.175), np.float64(0.2759), np.float64(0.1273), np.float64(0.1481), np.float64(0.1667)]
C2 Progressiv    [np.float64(0.5), np.float64(0.4483), np.float64(0.6), np.float64(0.4815), np.float64(0.5)]


## Gepaarter Vergleich

Entscheidend ist die Differenz **innerhalb** jeder Falte.

In [6]:
vergleiche = {}
if falten["B_direkt"] and falten["C2_progressiv"]:
    print("Progressiv gegenueber direkt:\n")
    vergleiche["C2_vs_B"] = fl.paired_comparison(
        falten["B_direkt"], falten["C2_progressiv"],
        "Direkt", "Progressiv", metric="Se")
    print()
    vergleiche["C2_vs_B_Sp"] = fl.paired_comparison(
        falten["B_direkt"], falten["C2_progressiv"],
        "Direkt", "Progressiv", metric="Sp", verbose=True)

if falten["A_zero_shot"] and falten["B_direkt"]:
    print("\nFine-Tuning gegenueber Zero-Shot:\n")
    vergleiche["B_vs_A"] = fl.paired_comparison(
        falten["A_zero_shot"], falten["B_direkt"],
        "Zero-Shot", "Direkt", metric="Se")

Progressiv gegenueber direkt:

  Gepaarter Vergleich (Se): Progressiv minus Direkt
    je Falte: [0.075, 0.069, 0.145, 0.037, 0.167]
    Mittelwert: +9.9 pp  (95 %-KI [+5.1, +14.7] pp)
    Progressiv besser in 5 von 5 Falten
    -> Vorteil fuer Progressiv statistisch gesichert.

  Gepaarter Vergleich (Sp): Progressiv minus Direkt
    je Falte: [0.001, -0.007, 0.002, -0.005, 0.0]
    Mittelwert: -0.2 pp  (95 %-KI [-0.5, +0.2] pp)
    Progressiv besser in 2 von 5 Falten
    -> Kein gesicherter Unterschied (Intervall schliesst null ein).

Fine-Tuning gegenueber Zero-Shot:

  Gepaarter Vergleich (Se): Direkt minus Zero-Shot
    je Falte: [0.1, 0.103, 0.309, 0.111, 0.0]
    Mittelwert: +12.5 pp  (95 %-KI [+2.6, +22.3] pp)
    Direkt besser in 4 von 5 Falten
    -> Vorteil fuer Direkt statistisch gesichert.


## Einordnung

In [7]:
v = vergleiche.get("C2_vs_B")
if v:
    print(f"Notebook 06 (eine Aufteilung): +5,6 pp, 36 Sturzfenster")
    print(f"Notebook 07 ({N_FOLDS} Falten):        "
          f"{v['mean_delta']*100:+.1f} pp, "
          f"{gepoolt['C2_progressiv']['TP'] + gepoolt['C2_progressiv']['FN']}"
          f" Sturzfenster")
    print()
    if v["significant"]:
        print("Der Unterschied ist ueber die Falten hinweg konsistent.")
    else:
        print("Der Unterschied ist statistisch nicht gesichert. Ein")
        print("Vorteil des progressiven Vorgehens laesst sich mit")
        print("diesen Daten nicht belegen; die geringe Zahl an")
        print("Sturzaufzeichnungen in UMAFall (157 insgesamt) duerfte")
        print("die Hauptursache sein.")

fl.save_results(run, {"gepoolt": gepoolt, "vergleiche": vergleiche,
                      "falten": falten, "n_folds": N_FOLDS,
                      "ziel_position": ZIEL_POSITION},
    config=fl.default_config(),
    notes=("Ersetzt Notebook 06. Fuenffache subjektunabhaengige "
           "Kreuzvalidierung mit gepaartem Vergleich."))

Notebook 06 (eine Aufteilung): +5,6 pp, 36 Sturzfenster
Notebook 07 (5 Falten):        +9.9 pp, 157 Sturzfenster

Der Unterschied ist ueber die Falten hinweg konsistent.
[gesichert] /content/drive/MyDrive/ML_Fall_Detection/results/07_progressiv_cv_2026-08-26_1218/results.json


PosixPath('/content/drive/MyDrive/ML_Fall_Detection/results/07_progressiv_cv_2026-08-26_1218/results.json')